In [8]:
#!/usr/bin/env python3
"""
Deduplicate an SDF by the *first* 'SMILES=' line of each record,
keeping the first occurrence and preserving full record formatting.
Outputs a clean SDF plus a CSV listing the retained SMILES.

Tested with RDKit 2023.09.
"""

from pathlib import Path
import csv

# --------- 変更しやすいパス設定 ---------
INPUT_SDF  = Path("../data/MoNA-GC-MS_filtered_ionization.sdf")  # 元 SDF
OUTPUT_SDF = Path("../data/MoNA_dedup.sdf") # ユニーク SDF
OUTPUT_CSV = Path("../data/MoNA_unique_smiles.csv")# SMILES 一覧
# ----------------------------------------

def first_smiles(record_lines):
    """Return the first SMILES=... string found in the record (empty if none)."""
    for ln in record_lines:
        if "SMILES=" in ln:
            return ln.split("SMILES=", 1)[1].strip()
    return ""

def main():
    seen   = set()
    smiles = []
    kept_records = []

    record_lines = []

    with INPUT_SDF.open(encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            if line.rstrip() == "$$$$":            # レコード終端
                record = "".join(record_lines) + "$$$$\n"
                smi = first_smiles(record_lines)
                if smi not in seen:
                    seen.add(smi)
                    kept_records.append(record)
                    smiles.append(smi)
                record_lines = []                  # 次のレコードへ
            else:
                record_lines.append(line)

    # ------ ファイル出力 ------
    OUTPUT_SDF.write_text("".join(kept_records), encoding="utf-8")
    print(f"✓ Unique SDF saved to: {OUTPUT_SDF}")

    with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["smiles"])
        w.writerows([[s] for s in smiles])
    print(f"✓ SMILES list saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


✓ Unique SDF saved to: ../data/MoNA_dedup.sdf
✓ SMILES list saved to: ../data/MoNA_unique_smiles.csv


6010